# Laya V7 — Native T4 INT4 × INT4 Tensor Core experiment (SM75)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V7_T4_True_INT4xINT4_TensorCore_Colab.ipynb)

**Why this notebook exists:** Your uploaded V6.1 results show **zero** INT4 backends and **zero** quantized modules on Tesla T4/SM75. The existing TorchAO *weight-only INT4* implementation requires BF16 kernels or MSLK variants unavailable on SM75. Merely lowering the speed threshold or installing MSLK will not solve it.

**New execution path:** actual Turing signed **INT4-activation × INT4-weight → INT32** Tensor Core MMA, with *dynamic per-token activation scaling* and *per-128-weight-group scales*. Packed INT4 matrices remain on GPU; the kernel does **not** materialize FP16 weights. This path requires the specialized [Turing Triton backend](https://github.com/Chennesxu/triton-turing), which exposes `tl.reinterpret_as_int4` for the Turing `m8n8k32` instruction. [NVIDIA CUTLASS documentation](https://github.com/NVIDIA/cutlass/blob/main/media/docs/cpp/implicit_gemm_convolution.md) describes the hardware.

**Important tradeoff:** dynamic INT4 activations add quantization error and kernel overhead. Achieving lower latency **and** keeping teacher agreement is not guaranteed. The experiment first checks numerical parity against a CPU/PyTorch INT4 reference, then benchmarks each actual Laya projection shape, and only installs kernels with faster operation **and** passing teacher fidelity. The independent audit is sealed until after selection. No model is automatically published.

**Installation:** the Turing fork is a native-build development dependency and can require a runtime restart after installation. If compilation or ABI compatibility fails, the notebook reports that error explicitly instead of reporting a fake INT4 speedup. Run this notebook in a fresh **Tesla T4** Colab runtime; do not mix standard Triton and the fork in an existing kernel.


In [ ]:
#@title 1. Install base dependencies and Turing-specific Triton (one-time fork build)
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" pandas numpy ninja cmake
import os,sys,subprocess,importlib
assert __import__("torch").cuda.is_available(),"Select a Tesla T4 GPU"
import torch
assert torch.cuda.get_device_capability()[0]==7, (
    "This notebook is specifically for Turing SM75; for L4/A100/H100 use V6")
import triton
import triton.language as tl
FORK_SHA="8da2085caa00c79cf7ce99c7927bb72135e36af1"
if not hasattr(tl,"reinterpret_as_int4"):
    print("Standard Triton has no SM75 int4 dot. Installing Turing fork at",FORK_SHA)
    # Intentionally pinned; native build may require matching CUDA/PyTorch.
    cmd=[sys.executable,"-m","pip","install","-v","--no-build-isolation",
         "git+https://github.com/Chennesxu/triton-turing.git@"+FORK_SHA]
    completed=subprocess.run(cmd,check=False)
    if completed.returncode:
        raise RuntimeError(
            "Turing Triton native build failed. No INT4 inference was attempted. "
            "Check compiler/CUDA errors above; avoid installing standard Triton "
            "over the custom fork in the same session.")
    raise RuntimeError(
        "Turing fork installed. **Runtime > Restart session**, then rerun "
        "cells from the top (do not reinstall if fork is now detected).")
print("SM75 Triton INT4 compiler primitive available:",triton.__version__)


In [ ]:
#@title 2. Model, experiment and native kernel settings
import sys,os,json,math,time,zipfile,copy,random,gc,importlib.util
from collections import Counter
from pathlib import Path
import torch,numpy as np,pandas as pd
from torch import nn
from torch.nn import functional as F
from huggingface_hub import HfApi,snapshot_download
from urllib.request import urlopen
from IPython.display import display

STARTING_REPO="vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
# To accelerate V5 rather than original Laya, provide the published
# standalone FP16 V5 Hugging Face repo in STARTING_REPO.
TRAIN_CASES=1200 #@param {type:"integer"}
SELECT_DECISIONS=256 #@param {type:"integer"}
GUARD_DECISIONS=128 #@param {type:"integer"}
AUDIT_DECISIONS=384 #@param {type:"integer"}
MAX_PROJECTION_TRIALS=48 #@param {type:"integer"}
MAX_ACCEPTED_INT4=24 #@param {type:"integer"}
MIN_OPERATOR_SPEEDUP=1.03 #@param {type:"number"}
MIN_END_TO_END_SPEEDUP=1.03 #@param {type:"number"}
MAX_NEW_SELECT_ERRORS=0 #@param {type:"integer"}
MAX_NEW_GUARD_ERRORS=0 #@param {type:"integer"}
MAX_JS=0.0015 #@param {type:"number"}
BENCH_REPEATS=14 #@param {type:"integer"}
SAVE_EVIDENCE=True #@param {type:"boolean"}
SEED=2062
OUT=Path("/content/Laya_V7_T4_True_INT4")
OUT.mkdir(parents=True,exist_ok=True)
torch.manual_seed(SEED);random.seed(SEED);np.random.seed(SEED)
try:
    from google.colab import userdata
    TOKEN=userdata.get("HF_TOKEN")
except Exception:TOKEN=os.environ.get("HF_TOKEN")
MODEL_REV=HfApi(token=TOKEN).model_info(STARTING_REPO).sha
ROOT=Path(snapshot_download(repo_id=STARTING_REPO,revision=MODEL_REV,token=TOKEN))
meta=json.loads((ROOT/"config.json").read_text())
assert meta.get("format")=="laya-1p2-full-standalone-v1"
assert meta.get("dtype_on_disk")=="float16"
print("Tesla T4 CUDA:",torch.cuda.get_device_name(0), "source:",MODEL_REV)


In [ ]:
#@title 3. Load Laya, true INT4xINT4 runtime, and CUDA timing helpers
def import_module(path,name):
    spec=importlib.util.spec_from_file_location(name,str(path))
    assert spec and spec.loader
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
    return module
source=import_module(ROOT/"modeling_laya_standalone.py","laya_sm75_source")
teacher,tokenizer=source.load_model(ROOT,device="cpu",local_files_only=True,compile_head=False)
teacher=teacher.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
try:
    if hasattr(teacher.encoder,"set_attn_implementation"):
        teacher.encoder.set_attn_implementation("sdpa")
    else:teacher.encoder.config._attn_implementation="sdpa"
except Exception as exc:print("SDPA advisory",str(exc)[:180])
assert len(teacher.encoder.layers)==28
V7_SHA="ab115350504b773374e00ddbe39755691af24342"
V6_SHA="a75e050dc9c3f9622a38854e7fe4b45ad3de3af6"
def fetch_module(sha,file,alias):
    target=Path("/content")/file
    with urlopen(f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{sha}/src/tinycenn_lm/{file}",
                 timeout=90) as response:
        target.write_bytes(response.read())
    return import_module(target,alias)
native=fetch_module(V7_SHA,"laya_sm75_true_int4_v7.py","laya_sm75_true_int4_v7")
benchmark=fetch_module(V6_SHA,"laya_native_int4_v6.py","laya_native_int4_v6")
native.require_native_turing()
print("INT4 backend capabilities",json.dumps(native.capability_report(),indent=2))


In [ ]:
#@title 4. Verify signed INT4 packing and hardware execution against reference
from torch import nn
# CPU / GPU packing bit-exactness
for axis in (0,1):
    qshape=(128,256)
    values=torch.randint(-8,8,qshape,device="cuda",dtype=torch.int8)
    packed=native.pack_signed_nibbles(values,axis)
    roundtrip=native.unpack_signed_nibbles(packed,axis)
    assert torch.equal(roundtrip,values),f"Signed nibble roundtrip failed axis={axis}"
print("Signed INT4 packed bitwise round-trip PASS")
# Check all four projection shape classes. Catch compilation/runtime problems now
# BEFORE full 28-layer search.
KERNEL_REPORTS=[]
for key in ("mlp.Wi","mlp.Wo","attn.Wqkv","attn.Wo"):
    name="encoder.layers.0."+key
    original=teacher.get_submodule(name)
    try:
        candidate=native.TuringINT4Linear(original).to("cuda")
        sample=torch.randn(7,original.in_features,dtype=torch.float16,device="cuda")
        actual=candidate(sample)
        reference=native.cpu_reference(
            sample,candidate.packed_weight,candidate.group_scales,candidate.bias)
        torch.testing.assert_close(actual,reference,rtol=0.04,atol=0.06)
        assert candidate.is_genuine_turing_s4s4
        KERNEL_REPORTS.append(dict(module=name,passes=True,
                                   max_abs=float((actual-reference).abs().max())))
        print("TURING INT4xINT4 HARDWARE PASS",name)
    except Exception as exc:
        KERNEL_REPORTS.append(dict(module=name,passes=False,error=repr(exc)[:800]))
        print("TURING INT4xINT4 FAILED",name,repr(exc)[:300])
    finally:
        if "candidate" in locals():del candidate
        gc.collect();torch.cuda.empty_cache()
(OUT/"kernel_preflight.json").write_text(json.dumps(KERNEL_REPORTS,indent=2))
if not any(x["passes"] for x in KERNEL_REPORTS):
    raise RuntimeError(
        "No real signed-INT4 Tensor Core projection passed its numerical "
        "reference check. Check Turing fork compilation before proceeding.")


In [ ]:
#@title 5. Hardware qualification: measured M=512 and M=2048, all shape classes
MICRO=[]
QUALIFIED={}
for item in KERNEL_REPORTS:
    if not item["passes"]:continue
    name=item["module"]
    source_linear=teacher.get_submodule(name)
    try:
        candidate=native.TuringINT4Linear(source_linear)
        speeds=[]
        for M in (128,512,2048):
            x=torch.randn(M,source_linear.in_features,
                          device="cuda",dtype=torch.float16)
            timing=benchmark.paired_cuda_latency(
                lambda:source_linear(x),lambda:candidate(x),
                repeats=BENCH_REPEATS)
            speeds.append(dict(m=M,**timing))
        has_gain=all(row["paired_speedup_median"]>=MIN_OPERATOR_SPEEDUP
                     and row["paired_speedup_p10"]>=1.0
                     for row in speeds if row["m"] in (512,2048))
        suffix=name.split(".",3)[-1]
        MICRO.append(dict(projection=suffix,speeds=speeds,qualified=has_gain))
        if has_gain:QUALIFIED[suffix]=True
        print(suffix,"qualified",has_gain,
              "speedups",[round(x["paired_speedup_median"],3) for x in speeds])
    except Exception as exc:
        MICRO.append(dict(projection=name,error=repr(exc)[:500],qualified=False))
        print("Microbench failed",name,repr(exc)[:200])
    finally:
        if "candidate" in locals():del candidate
(OUT/"int4_hardware_speed.json").write_text(json.dumps(MICRO,indent=2))
print("Actually faster INT4 projection shape classes:",list(QUALIFIED))


In [ ]:
#@title 6. Case-disjoint selection, guard, and sealed independent audit
from datasets import load_dataset
def parse_json(x):
    if isinstance(x,str):
        try:return json.loads(x)
        except Exception:return x
    return x
def prepare_items(cases,needed):
    grouped=[]
    for row in cases:
        questions=parse_json(row.get("questions"))
        state=parse_json(row.get("state"))
        if not isinstance(questions,dict):continue
        choices=[]
        for question in questions.values():
            try:
                normalized=source.normalize_question(parse_json(question))
                ids,marks=source.build_sequence(tokenizer,state,normalized,max_len=512,
                          head_max_len=192,truncate_left=isinstance(state,list))
                if len(marks)<2:continue
                choices.append(dict(ids=ids,markers=marks,
                                    qtype=source.QTYPES[normalized["t"]]))
            except (TypeError,ValueError,KeyError,IndexError):continue
        if choices:grouped.append(choices)
    output=[]
    for depth in range(max(map(len,grouped),default=0)):
        for group in grouped:
            if depth<len(group):
                output.append(group[depth])
                if len(output)>=needed:return output
    return output
def collate(items,batch_size=4,pad_to=None):
    batches=[]
    for i in range(0,len(items),batch_size):
        b=source.collate_items(items[i:i+batch_size],int(tokenizer.pad_token_id))
        if pad_to is not None:
            missing=pad_to-b["input_ids"].shape[1]
            if missing<0:raise ValueError("Example too long")
            b["input_ids"]=F.pad(b["input_ids"],(0,missing),
                     value=int(tokenizer.pad_token_id))
            b["attention_mask"]=F.pad(b["attention_mask"],(0,missing))
        batches.append({k:v.to("cuda") for k,v in b.items()})
    return batches
dataset=list(load_dataset("LocalLLaMA/typed-decisions","all",split=f"train[:{TRAIN_CASES}]"))
random.Random(SEED).shuffle(dataset)
cuts=[0,int(len(dataset)*.10),int(len(dataset)*.40),
      int(len(dataset)*.55),len(dataset)]
sel_items=prepare_items(dataset[cuts[1]:cuts[2]],SELECT_DECISIONS)
guard_items=prepare_items(dataset[cuts[2]:cuts[3]],GUARD_DECISIONS)
audit_items=prepare_items(dataset[cuts[3]:],AUDIT_DECISIONS)
assert [len(sel_items),len(guard_items),len(audit_items)]==[
    SELECT_DECISIONS,GUARD_DECISIONS,AUDIT_DECISIONS]
SEL,GUARD,AUDIT=[collate(items) for items in (sel_items,guard_items,audit_items)]
@torch.inference_mode()
def infer(model,b):
    with torch.autocast("cuda",dtype=torch.float16):
        o=model(**b)
    return o[0] if isinstance(o,(tuple,list)) else o
@torch.inference_mode()
def teacher_logits(batches):
    return [infer(teacher,b).float().cpu() for b in batches]
REF_SEL,REF_GUARD,REF_AUDIT=[teacher_logits(x) for x in (SEL,GUARD,AUDIT)]
@torch.inference_mode()
def metric(model,batches,truth):
    count=errors=0;total_js=0.
    wrong=[]
    for batch,teacher_out in zip(batches,truth):
        student_out=infer(model,batch).float().cpu()
        active=batch["marker_mask"].bool().cpu()
        y=student_out.masked_fill(~active,-1e4)
        t=teacher_out.masked_fill(~active,-1e4)
        probs=y.softmax(-1).clamp_min(1e-9)
        ref=t.softmax(-1).clamp_min(1e-9)
        mix=0.5*(probs+ref)
        js=.5*(probs*(probs/mix).log()).sum(-1)+.5*(ref*(ref/mix).log()).sum(-1)
        is_wrong=(probs.argmax(-1)!=ref.argmax(-1))
        wrong.extend(is_wrong.tolist())
        errors+=int(is_wrong.sum());count+=len(is_wrong);total_js+=float(js.sum())
    return dict(n=count,errors=errors,agreement=1-errors/count,mean_js=total_js/count,
                error_mask=wrong)


In [ ]:
#@title 7. Accuracy-first true T4 INT4 substitution with strict rollback
student=copy.deepcopy(teacher).eval().requires_grad_(False)
chosen=[];trials=[]
NAMES=[f"encoder.layers.{i}.{suffix}" for i in range(28)
       for suffix in ("mlp.Wi","mlp.Wo","attn.Wqkv","attn.Wo")]
baseline_select=metric(student,SEL,REF_SEL)
baseline_guard=metric(student,GUARD,REF_GUARD)
current_select,current_guard=baseline_select,baseline_guard
for name in NAMES[:MAX_PROJECTION_TRIALS]:
    if len(chosen)>=MAX_ACCEPTED_INT4:break
    suffix=name.split(".",3)[-1]
    if suffix not in QUALIFIED:continue
    old=student.get_submodule(name)
    trial=dict(name=name,accepted=False)
    try:
        candidate=native.TuringINT4Linear(old)
        parent,_,component=name.rpartition(".")
        setattr(student.get_submodule(parent),component,candidate)
        score_sel=metric(student,SEL,REF_SEL)
        score_guard=metric(student,GUARD,REF_GUARD)
        new_sel=sum(x and not y for x,y in zip(
            score_sel["error_mask"],current_select["error_mask"]))
        new_guard=sum(x and not y for x,y in zip(
            score_guard["error_mask"],current_guard["error_mask"]))
        keep=(new_sel<=MAX_NEW_SELECT_ERRORS and
              new_guard<=MAX_NEW_GUARD_ERRORS and
              score_sel["errors"]<=current_select["errors"] and
              score_guard["errors"]<=current_guard["errors"] and
              score_sel["mean_js"]<=MAX_JS and score_guard["mean_js"]<=MAX_JS)
        trial.update(accepted=bool(keep),select=score_sel["agreement"],
                     guard=score_guard["agreement"],new_select_errors=new_sel,
                     new_guard_errors=new_guard)
        if keep:
            chosen.append(name)
            current_select,current_guard=score_sel,score_guard
            print("ACCEPT T4 native W4A4",name,
                  "select",score_sel["agreement"],"guard",score_guard["agreement"])
        else:
            setattr(student.get_submodule(parent),component,old)
    except Exception as exc:
        setattr(student.get_submodule(parent),component,old)
        trial["error"]=repr(exc)[:450]
        print("Turing INT4 candidate rejected",name,trial["error"][:180])
    trials.append(trial)
    if len(trials)%4==0:
        (OUT/"layer_trials.json").write_text(json.dumps(trials,indent=2))
(OUT/"layer_trials.json").write_text(json.dumps(trials,indent=2))
print("True SM75 INT4 Tensor Core layers kept:",len(chosen))


In [ ]:
#@title 8. Sealed audit + fair end-to-end acceleration assessment
FINAL_SEL=metric(student,SEL,REF_SEL)
FINAL_GUARD=metric(student,GUARD,REF_GUARD)
FINAL_AUDIT=metric(student,AUDIT,REF_AUDIT)
AUDIT_PASS=(FINAL_AUDIT["agreement"]>=0.99 and FINAL_AUDIT["mean_js"]<=MAX_JS)
print("Audit gate",AUDIT_PASS,
      {k:v for k,v in FINAL_AUDIT.items() if k!="error_mask"})
BENCH=[]
for B in (1,4):
    selected=sel_items[:B]
    minimum=max(len(item["ids"]) for item in selected)
    for T in (128,256,512):
        if minimum>T:continue
        b=collate(selected,batch_size=B,pad_to=T)[0]
        if chosen:
            speed=benchmark.paired_cuda_latency(
                lambda:infer(teacher,b),lambda:infer(student,b),
                repeats=BENCH_REPEATS)
            row=dict(batch=B,tokens=T,**speed,
                     verified_operator_count=len(chosen),identity_control=False)
        else:
            base=benchmark.cuda_events_ms(lambda:infer(teacher,b),
                                           repeats=BENCH_REPEATS)
            row=dict(batch=B,tokens=T,fp16_median_ms=base["median_ms"],
                     candidate_median_ms=base["median_ms"],
                     paired_speedup_median=1.0,paired_speedup_p10=1.0,
                     verified_operator_count=0,identity_control=True)
        BENCH.append(row)
        print("B",B,"T",T,"speedup",row["paired_speedup_median"])
bench=pd.DataFrame(BENCH)
bench.to_csv(OUT/"end_to_end_v7.csv",index=False)
display(bench)
targets=bench[(bench.tokens==512)&bench.batch.isin((1,4))]
SPEED_PASS=(len(chosen)>0 and len(targets)==2
            and bool((targets.paired_speedup_median>=MIN_END_TO_END_SPEEDUP).all())
            and bool((targets.paired_speedup_p10>=1.0).all()))
VERIFIED=bool(chosen) and SPEED_PASS and AUDIT_PASS
print("TRUE T4 INT4 SPEED+FIDELITY VERIFIED:",VERIFIED)


In [ ]:
#@title 9. ZIP all evidence and preserve exact compiler/model revisions
record=dict(source_repo=STARTING_REPO,source_sha=MODEL_REV,
   gpu=torch.cuda.get_device_name(0),triton=__import__("triton").__version__,
   turing_fork_sha=FORK_SHA,backend_sha=V7_SHA,
   installed_true_int4_modules=chosen,installed_count=len(chosen),
   activation_precision="dynamic signed 4-bit per token",
   weight_precision="signed 4-bit, group128 scales",
   gpu_matmul="native SM75 INT4xINT4→INT32 m8n8k32",
   accuracy_select={k:v for k,v in FINAL_SEL.items() if k!="error_mask"},
   accuracy_guard={k:v for k,v in FINAL_GUARD.items() if k!="error_mask"},
   sealed_audit={k:v for k,v in FINAL_AUDIT.items() if k!="error_mask"},
   audit_pass=bool(AUDIT_PASS),speed_gate_pass=bool(SPEED_PASS),
   independently_verified=bool(VERIFIED),
   performance=BENCH,
   note="Experiment only; no standalone HF INT4 loader or speedup claim without successful CUDA benchmarks.")
(OUT/"summary_sm75_v7.json").write_text(json.dumps(record,indent=2))
archive=OUT/"Laya_V7_T4_True_INT4xINT4_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for file in OUT.iterdir():
        if file.is_file() and file!=archive:z.write(file,file.name)
print("Evidence:",archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass


## Validation conditions and limitations

A successful run requires three independently verified facts: (1) the specialized Triton fork compiled and emitted genuine SM75 INT4 Tensor Core instructions; (2) numeric output of each projection agrees with a PyTorch signed-INT4 reconstruction reference; (3) the selected modules pass source-case-disjoint fidelity and model-level speed gates.

**Important:** The reference check confirms correct W4A4 integer arithmetic, not closeness to the original FP16 teacher. Accuracy may drop from activation quantization, and accepting zero modules is a valid negative result. The Turing Triton backend is experimental third-party code and may be incompatible with a given Colab/CUDA/PyTorch combination.

No existing Hugging Face model is modified or automatically published.
